# Hyperparameter Tuning

Welcome to this focused, practical, and revision-friendly notebook on **Hyperparameter Tuning**.
This notebook focuses on the concepts directly required to understand and use hyperparameter tuning to improve machine learning models.
We will use previously learned models (Logistic Regression, KNN, Decision Tree, Random Forest, SVM) to demonstrate these concepts.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Scikit-learn imports will be introduced exactly when needed


# 1. What is Hyperparameter Tuning?

### Concept
Every machine learning algorithm has two types of variables that dictate how it behaves: **Parameters** and **Hyperparameters**.

* **What is a parameter?**
  A variable that the model *learns* on its own from the training data. You do not set these manually.

* **What is a hyperparameter?**
  A variable that the user (you) sets *before* the learning process begins. These govern how the model learns or what its architecture looks like.

* **Why hyperparameters matter**
  Hyperparameters control the complexity, flexibility, and generalization ability of a model. Setting them correctly can mean the difference between a model that underfits (too simple) or overfits (too complex).

* **Why default hyperparameters are not always optimal**
  Scikit-learn provides sensible defaults, but they are generic. Every dataset is unique, meaning the optimal configuration for your specific problem will likely differ from the default settings.

> **Key Idea**: Hyperparameter tuning is the systematic process of searching for the optimal combination of hyperparameters to maximize your model's performance on unseen data.

### Simple Examples

* **KNN**: `n_neighbors` (The 'K'. How many neighbors to consider?)
* **Decision Tree**: `max_depth` (How deep can the tree grow?)
* **Random Forest**: `n_estimators` (How many individual decision trees to build?)
* **Logistic Regression**: `C` (Inverse of regularization strength)
* **SVM**: `C` (Regularization parameter), `gamma` (Kernel coefficient)


# 2. Parameters vs Hyperparameters

Here is a clear comparison to help you distinguish between the two:

| Parameters | Hyperparameters |
| :--- | :--- |
| Learned automatically during training | Set manually before/during training |
| Adjusted by the optimization algorithm (e.g., Gradient Descent) | Searched via tuning (e.g., Grid Search, Random Search) |
| **Examples:** | **Examples:** |
| Model weights in Neural Networks | K in KNN (`n_neighbors`) |
| Coefficients in Linear Regression | Tree depth (`max_depth`) |
| Intercept term | Number of trees in a forest (`n_estimators`) |

> **Remember:** If it is something you pass as an argument when initializing the model, it is a hyperparameter.


# 3. Why Hyperparameter Tuning is Needed

The general workflow for machine learning looks like this:

**Default settings &rarr; Train model &rarr; Evaluate &rarr; Tune hyperparameters &rarr; Validate &rarr; Select better configuration**

Hyperparameter tuning is essential because it is the primary way to navigate the **bias-variance tradeoff**.

* **Underfitting (High Bias):** The model is too simple to capture the underlying patterns (e.g., decision tree with `max_depth=1`). We need to tune hyperparameters to allow *more* flexibility.
* **Good Fit:** The model captures the underlying pattern without memorizing noise.
* **Overfitting (High Variance):** The model memorizes the training data and noise (e.g., decision tree with `max_depth=None` on a noisy dataset). We need to tune hyperparameters to *regularize* or restrict the model.

Let's look at a practical example where default hyperparameters might fail.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Generate a synthetic dataset
X, y = make_classification(n_samples=500, n_features=2, n_informative=2, n_redundant=0, 
                           n_clusters_per_class=1, flip_y=0.1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Using default hyperparameters (n_neighbors=5)
knn_default = KNeighborsClassifier()
knn_default.fit(X_train, y_train)

print(f"Default KNN Train Accuracy: {accuracy_score(y_train, knn_default.predict(X_train)):.4f}")
print(f"Default KNN Test Accuracy: {accuracy_score(y_test, knn_default.predict(X_test)):.4f}")


# 4. Validation Data and Tuning

> **Important**: Never use the final test set to choose your hyperparameters!

If you repeatedly train a model, test it on the test set, tweak the hyperparameters, and test it on the test set again, you are indirectly leaking information from the test set into your model. Your model will **overfit to the test set**.

### The Proper Workflow

**Training Data &rarr; Cross Validation &rarr; Hyperparameter Tuning &rarr; Best Model &rarr; Final Test Evaluation**

1. **Test Set**: Keep it completely locked away. Untouched.
2. **Training Set**: Split it further (usually via cross-validation). 
3. **Validation Set(s)**: Used repeatedly to evaluate the model while you tweak hyperparameters.
4. Only after you are completely satisfied with your tuned model do you evaluate it on the untouched Test Set.


# 5. Manual Hyperparameter Tuning

Let's demonstrate a simple manual experiment using KNN. 
We will test several values for `n_neighbors` manually using Cross-Validation.


In [ ]:
from sklearn.model_selection import cross_val_score

k_values = [1, 3, 5, 7, 9, 11, 15]

results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # Using 5-fold cross validation on the training data only
    cv_scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')

    results.append({
        'K': k,
        'Mean Accuracy': cv_scores.mean(),
        'Standard Deviation': cv_scores.std()
    })

# Create a results DataFrame
manual_tuning_df = pd.DataFrame(results)
display(manual_tuning_df)


Now let's visualize the results to find the sweet spot.


In [ ]:
plt.figure(figsize=(8, 5))
plt.errorbar(manual_tuning_df['K'], manual_tuning_df['Mean Accuracy'], yerr=manual_tuning_df['Standard Deviation'], fmt='-o', capsize=5)
plt.title('K vs CV Accuracy')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Mean CV Accuracy')
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


### Why Manual Tuning is Insufficient
Manual tuning is easy for a single hyperparameter (like $K$). But what if we also want to tune:
* `weights` (uniform vs distance)
* `metric` (euclidean vs manhattan)
* `p` (power parameter for Minkowski metric)

If we have 7 values for K, 2 for weights, and 2 for metric, that's $7 	imes 2 	imes 2 = 28$ combinations! Writing nested loops for all these combinations is tedious and error-prone. We need an automated approach.


# 6. Grid Search

### Concept
* **What is Grid Search?** 
  It is a technique that exhaustively searches through a manually specified subset of the hyperparameter space.
* **How it works:**
  You define a dictionary (a "grid") of hyperparameters and the values you want to test. Grid Search trains and evaluates a model for *every possible combination* using cross-validation.
* **Advantages:**
  Guaranteed to find the best combination *within the grid you provided*.
* **Disadvantages:**
  Computationally very expensive (the "Curse of Dimensionality"). As you add more hyperparameters and values, the number of combinations multiplies exponentially.


In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Define the model
knn_grid = KNeighborsClassifier()

# 2. Define the parameter grid
param_grid = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

# 3. Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=knn_grid,
    param_grid=param_grid,
    cv=5,                 # 5-fold cross validation
    scoring="accuracy",   # Metric to optimize
    n_jobs=-1             # Use all available CPU cores
)

# 4. Run the search on the training data
grid_search.fit(X_train, y_train)

# 5. Show best results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Best estimator: {grid_search.best_estimator_}")


# 7. Understanding GridSearchCV Results

The `GridSearchCV` object contains several important attributes after fitting:

* `best_params_`: The dictionary of parameters that gave the best results.
* `best_score_`: The mean cross-validated score of the `best_estimator_`.
* `best_estimator_`: The actual instantiated model, trained with the best parameters on the *entire* training dataset, ready to make predictions.
* `cv_results_`: A dictionary with detailed results of every single experiment.

Let's convert `cv_results_` into a Pandas DataFrame to analyze the most important columns.


In [ ]:
# Convert cv_results_ to a DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

# Select only the most important columns for readability
important_columns = ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
parsed_results = results_df[important_columns].copy()

# Sort by the score to see the best combinations at the top
parsed_results = parsed_results.sort_values(by='rank_test_score').reset_index(drop=True)

display(parsed_results.head(10))


# 8. Randomized Search

### Concept
* **What is Randomized Search?** 
  Instead of trying *every* combination, Randomized Search samples a fixed number of parameter settings from specified distributions.
* **Difference from Grid Search:**
  Grid search is exhaustive. Randomized search samples randomly.
* **Why it can be faster:**
  You define exactly how many iterations (`n_iter`) it will run. If the search space is huge, Grid Search might take days. Randomized Search will run in a fixed amount of time.
* **Why it can explore a larger space efficiently:**
  Not all hyperparameters are equally important. Grid Search spends equal time on less important hyperparameters. Random tracking explores a wider variety of values for the important parameters faster.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

# 1. Define model
rf = RandomForestClassifier(random_state=42)

# 2. Define the parameter distribution using scipy.stats where helpful
param_dist = {
    "n_estimators": randint(50, 300),  # Random integer between 50 and 300
    "max_depth": [None, 3, 5, 10, 20],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "criterion": ["gini", "entropy"]
}

# 3. Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,           # Number of random combinations to try
    cv=5,
    scoring="accuracy",
    random_state=42,     # So we get the same random combinations every time
    n_jobs=-1
)

# 4. Run search
random_search.fit(X_train, y_train)

# 5. Show best results
print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")

# Finally evaluate on test set
best_rf = random_search.best_estimator_
rf_test_acc = accuracy_score(y_test, best_rf.predict(X_test))
print(f"Final test performance: {rf_test_acc:.4f}")


# 9. Grid Search vs Random Search

| Feature | Grid Search | Random Search |
| :--- | :--- | :--- |
| **Strategy** | Tests all possible combinations | Tests a random sample of combinations |
| **Search space** | Exhaustive (discrete grid) | Sampled (can use continuous distributions) |
| **Computational cost**| Can be extremely high | Usually much lower (controlled via `n_iter`) |
| **Best use case** | Small parameter space, few hyperparameters | Large parameter space, many hyperparameters |

> **Key Takeaway**: Use Grid Search when you have a small number of parameters and you want to be completely sure you found the absolute best combination within your grid. Use Randomized Search for complex models (like Random Forests or Gradient Boosting) where the number of parameters is large.


# 10. Cross Validation During Tuning

Why should hyperparameter tuning be combined with cross-validation?

If we evaluate hyperparameter performance on just one simple Train/Validation split, our "optimal" hyperparameters might just be overfitting to that specific Validation split!

**Workflow with CV:**
**Hyperparameter combo &rarr; Split into folds &rarr; Train/validate repeatedly &rarr; Average performance &rarr; Select best configuration**

By averaging the score across 5 or 10 folds, we get a highly robust and reliable estimate of how well that specific hyperparameter combination will generalize. Both `GridSearchCV` and `RandomizedSearchCV` do this automatically.


# 11. Choosing the Scoring Metric

When using setup objects like `GridSearchCV`, the `scoring` parameter dictates what metric the algorithm is trying to maximize.

> **Important**: The best hyperparameters depend on the metric being optimized!

Common metrics:
* `accuracy`
* `precision`
* `recall`
* `f1`
* `roc_auc`

**Example: Imbalanced Classification**
If you have a dataset where 99% of samples are Class 0 and 1% are Class 1 (like fraud detection), optimizing for `accuracy` will lead the tuner to select hyperparameters that simply predict Class 0 all the time. Instead, you would want to optimize for `recall` or `roc_auc`.


# 12. Hyperparameter Tuning — Logistic Regression

Let's tune a Logistic Regression model. Logistic Regression is relatively simple, so Grid Search is appropriate.


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Inverse of regularization strength
    'penalty': ['l2', 'l1'],              # Regularization type
    'solver': ['liblinear', 'saga']       # Solvers that support both l1 and l2
}

grid_lr = GridSearchCV(log_reg, param_grid_lr, cv=5, scoring='accuracy')
grid_lr.fit(X_train, y_train)

print(f"Best Logistic Regression configuration: {grid_lr.best_params_}")
print(f"Best CV Score: {grid_lr.best_score_:.4f}")


# 13. Hyperparameter Tuning — KNN

We previously tuned KNN manually and sequentially. Here is the formal `GridSearchCV` implementation.


In [ ]:
knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(knn, param_grid_knn, cv=5, scoring='accuracy')
grid_knn.fit(X_train, y_train)

print(f"Best KNN configuration: {grid_knn.best_params_}")
print(f"Best CV Score: {grid_knn.best_score_:.4f}")


*Interpretation*: The grid search not only found the best `K`, but also matched it with the best distance metric and voting mechanism (`weights`).


# 14. Hyperparameter Tuning — Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

param_grid_dt = {
    'max_depth': [None, 3, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}

grid_dt = GridSearchCV(dt, param_grid_dt, cv=5, scoring='accuracy')
grid_dt.fit(X_train, y_train)

print(f"Best Decision Tree configuration: {grid_dt.best_params_}")


**How these parameters control model complexity:**
* `max_depth`: Limits the depth of the tree. Smaller depth = less complexity (prevents overfitting).
* `min_samples_split`: Minimum number of samples required to split a node. Higher number = smoother boundaries.
* `min_samples_leaf`: Minimum number of samples required to be at a leaf node. Higher number = prevents creating leaves for outliers.
* `criterion`: Function to measure the quality of a split.


# 15. Hyperparameter Tuning — Random Forest

Random Forests have a massive parameter space. Running an exhaustive Grid Search would take too long, which is why we use `RandomizedSearchCV`.


In [ ]:
rf_tune = RandomForestClassifier(random_state=42)

param_dist_rf = {
    'n_estimators': randint(50, 200),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt', 'log2']
}

random_rf = RandomizedSearchCV(rf_tune, param_dist_rf, n_iter=15, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
random_rf.fit(X_train, y_train)

print(f"Best Random Forest configuration: {random_rf.best_params_}")


**Why rely on RandomizedSearchCV?** 
If we used a Grid Search using the equivalent ranges, we'd be training tens of thousands of models. Random Search covers the bounds of our distribution in just 15 iterations.


# 16. Hyperparameter Tuning — SVM

Support Vector Machines (SVMs) are highly sensitive to their hyperparameters `C` and `gamma`.


In [ ]:
from sklearn.svm import SVC

svm_model = SVC(random_state=42)

# Small grid due to computational expense of SVMs
param_grid_svm = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 1],
    'kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(svm_model, param_grid_svm, cv=5, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_train, y_train)

print(f"Best SVM configuration: {grid_svm.best_params_}")


# 17. Pipeline + Hyperparameter Tuning

> **Important**: This is a critical practical section!

In the real world, you almost always need to preprocess your data (scaling, encoding, etc.). If you apply preprocessing using `StandardScaler` *before* doing cross-validation, you are leaking information!

The proper way to combine preprocessing and hyperparameter tuning is by using a `Pipeline`. When we use a Pipeline inside `GridSearchCV`, the preprocessing (e.g., scaling) happens **independently on the training portion of every cross-validation fold**.

**Workflow:**
StandardScaler &rarr; PCA (optional) &rarr; Model (all inside a Pipeline)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a pipeline connecting Scaler and Model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

# For tuning inside a pipeline, you must use the format: step_name__parameter_name
param_grid_pipe = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear']
}

grid_pipe = GridSearchCV(pipeline, param_grid_pipe, cv=5)
grid_pipe.fit(X_train, y_train)

print(f"Best Pipeline configuration: {grid_pipe.best_params_}")


# 18. Avoiding Data Leakage During Tuning

Data leakage happens when information from outside the training dataset is used to create the model.

> **Common Mistake (Incorrect):**
> Scale complete dataset &rarr; GridSearchCV
> *Why is it wrong?* The scaler learned the mean and variance of the validation folds during the scaling step *before* CV started. Thus, the validation fold is no longer truly "unseen".

> **Correct Approach:**
> Pipeline &rarr; GridSearchCV &rarr; Cross-validation
> *Why is it correct?* The pipeline protects against leakage because scaling is recalculated **only on the training folds** repeatedly for every new split within CV.


# 19. Nested Cross Validation — Concept Only

When you use `GridSearchCV`, the best score reported (`best_score_`) is often slightly optimistic because the hyperparameters were explicitly tuned to look good on those validation folds. 

If you want a truly unbiased estimate of how well the tuning process itself is performing, we use **Nested Cross-Validation**.

* **Inner Loop:** Does the hyperparameter tuning (GridSearchCV). Finds the best hyperparameter combination on subsets of data.
* **Outer Loop:** Repeatedly evaluates the *entire inner loop procedure* on new test sets.

This is computationally expensive and conceptually difficult, but highly rigorous.


# 20. Overfitting During Hyperparameter Tuning

Can you overfit during hyperparameter tuning? **Yes!**

**Too many experiments &rarr; Validation performance looks better &rarr; True generalization may not improve**

If you test 10,000 parameter combinations on your cross-validation folds, eventually you will find a combination that gets a fantastic score purely by statistical accident (memorizing the validation folds). 

This is exactly why **the final test set should be used only once at the end.** It guarantees an unbiased check on whether your tuning was genuinely successful or if you just overfitted the validation sets.


# 21. Complete End-to-End Hyperparameter Tuning Project

Let's build one complete, realistic classification project utilizing pipeless and proper test set splits.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Load dataset
data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_bc = pd.Series(data.target)

# 2. Train/test split - Lock away the test set!
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

# 3. Define pipelines
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

# 4. Baseline models (Evaluate without tuning on test set)
pipe_lr.fit(X_train_bc, y_train_bc)
lr_base_pred = pipe_lr.predict(X_test_bc)
lr_base_acc = accuracy_score(y_test_bc, lr_base_pred)

pipe_rf.fit(X_train_bc, y_train_bc)
rf_base_pred = pipe_rf.predict(X_test_bc)
rf_base_acc = accuracy_score(y_test_bc, rf_base_pred)

print(f"Baseline Logistic Regression Test Accuracy: {lr_base_acc:.4f}")
print(f"Baseline Random Forest Test Accuracy: {rf_base_acc:.4f}")

# 6. Define parameter grids & distributions
param_grid_lr_bc = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2'] # Keeping it simple for L2 solver
}

param_dist_rf_bc = {
    'clf__n_estimators': randint(50, 250),
    'clf__max_depth': [None, 5, 10, 20],
    'clf__min_samples_split': randint(2, 10)
}

# 7 & 8. Run searches
grid_lr_bc = GridSearchCV(pipe_lr, param_grid_lr_bc, cv=5, scoring='accuracy')
grid_lr_bc.fit(X_train_bc, y_train_bc)

random_rf_bc = RandomizedSearchCV(pipe_rf, param_dist_rf_bc, n_iter=20, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
random_rf_bc.fit(X_train_bc, y_train_bc)

# 9. Evaluate best CV score
print(f"\nTuned Logistic Regression Best CV Score: {grid_lr_bc.best_score_:.4f}")
print(f"Tuned Random Forest Best CV Score: {random_rf_bc.best_score_:.4f}")

# 10. Evaluate final selected model on untouched test set
lr_tuned_pred = grid_lr_bc.predict(X_test_bc)
rf_tuned_pred = random_rf_bc.predict(X_test_bc)

# 11-16. Compare baseline vs tuned
print("\n--- Final Test Set Evaluation ---")
print(f"Tuned Logistic Regression Accuracy: {accuracy_score(y_test_bc, lr_tuned_pred):.4f}")
print(f"Tuned Random Forest Accuracy: {accuracy_score(y_test_bc, rf_tuned_pred):.4f}")

print(f"\nTuned LR Precision: {precision_score(y_test_bc, lr_tuned_pred):.4f}")
print(f"Tuned LR Recall: {recall_score(y_test_bc, lr_tuned_pred):.4f}")
print(f"Tuned LR F1 Score: {f1_score(y_test_bc, lr_tuned_pred):.4f}")
print(f"Tuned LR ROC AUC: {roc_auc_score(y_test_bc, grid_lr_bc.predict_proba(X_test_bc)[:, 1]):.4f}")

# 17. Confusion Matrix
cm = confusion_matrix(y_test_bc, lr_tuned_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)


### 18. Did tuning actually improve the model?
*(Run the code block below to see visually)*
Notice that sometimes tuning simple models with strong default parameters might result in little to no real-world gain on the test set, demonstrating that Scikit-Learn defaults are very mathematically sound for typical tabular data, but tuning offers granular optimization.


# 22. Tuning Results Visualization

Visualizing the baseline vs tuned performance can highlight where actual improvements occurred.


In [ ]:
# Visualizing Baseline vs Tuned Performance
labels = ['LR Baseline', 'LR Tuned', 'RF Baseline', 'RF Tuned']
accuracies = [lr_base_acc, accuracy_score(y_test_bc, lr_tuned_pred), rf_base_acc, accuracy_score(y_test_bc, rf_tuned_pred)]

plt.figure(figsize=(8,5))
sns.barplot(x=labels, y=accuracies, palette='viridis')
plt.ylim(0.9, 1.0)
plt.title('Baseline vs Tuned Model Accuracy on Test Set')
plt.ylabel('Accuracy')
plt.show()

# Visualize Confusion Matrix for best model
disp.plot(cmap='Blues')
plt.title("Confusion Matrix: Tuned Logistic Regression")
plt.show()


# 23. Practical Guidelines for Hyperparameter Tuning

✔️ **1. Start with a baseline:** Never start tuning immediately. Always check how a default model performs first.
✔️ **2. Choose the right metric:** Tuning for `accuracy` when data is imbalanced is a fatal flaw. Select carefully based on business goals.
✔️ **3. Use cross-validation:** Avoid 1-split validation sets as they lead to statistical noise dominating the decision.
✔️ **4. Tune a reasonable parameter range:** Do not test 50 variations of `max_depth`. Use domain knowledge.
✔️ **5. Use Grid Search for small spaces:** Perfect for Logistic Regression, small KNN arrays.
✔️ **6. Use Randomized Search for larger spaces:** Essential for Random Forests and XGBoost/Gradient Boosters.
✔️ **7. Keep preprocessing inside Pipeline:** Avoid silent, subtle data leakage.
✔️ **8. Avoid tuning on the test set:** The test set must be used zero times during the tuning loop.
✔️ **9. Compare baseline vs tuned model:** Validate that your 30 minutes of CPU time was actually worth 0.5% gain.
✔️ **10. Use the final test set only once:** Evaluate your `best_estimator_` and report that number truthfully.


# 24. Common Hyperparameter Tuning Mistakes

1. **Tuning on test data**
   * *Problem:* You accidentally cross-validate or check test scores during tuning loops.
   * *Why:* Causes information leakage; true performance will collapse in production.
   * *Fix:* Strict Train-Test split. Put the Test set in a completely different variable.

2. **Huge parameter grids without reason**
   * *Problem:* Testing trees with depth [1, 2, ..., 100].
   * *Why:* Massively explodes computing time without theoretical benefits. Note: large trees overfit rapidly.
   * *Fix:* Test broad ranges first (e.g. `[10, 50, None]`).

3. **Wrong scoring metric**
   * *Problem:* Tuning for accuracy in medical diagnosis (rare class).
   * *Why:* The tuner will deliberately pick bad hyperparameters that ignore sickness just to max accuracy.
   * *Fix:* Use `scoring='f1'` or `roc_auc`.

4. **No cross-validation**
   * *Problem:* Using a single train/validation split because CV takes too long.
   * *Why:* Hyperparameter combinations will easily overfit to that single Validation split.
   * *Fix:* Always use CV unless working with deep learning / massive datasets.

5. **Data leakage**
   * *Problem:* `X_scaled = Scaler.fit_transform(X)`, then entering CV.
   * *Why:* Validation folds during CV contain information from the training folds' mean/variance.
   * *Fix:* Use `Pipeline`.

6. **Too many experiments**
   * *Problem:* Running RandomizedSearchCV for 500,000 iterations.
   * *Why:* Leads to "Multiple Comparison Fallacy" -> eventual overfitting.
   * *Fix:* Limit the number of iterations logically. 

7. **Ignoring computational cost**
   * *Problem:* Attempting exhaustive GridSearch on a server without considering time constraints.
   * *Why:* Script might take 90 days to run.
   * *Fix:* Use RandomizedSearch, or Optuna/Bayesian optimization methods for massive models.

8. **Optimizing one metric blindly**
   * *Problem:* Focusing ONLY on maximizing `accuracy` while Precision drops drastically.
   * *Why:* Real-world tasks usually demand balance.
   * *Fix:* Examine the classification report on the best model, not just the single metric number.

9. **Using inconsistent preprocessing**
   * *Problem:* Baseline used Scaler A, tuned model used Scaler B.
   * *Why:* Invalid comparison.
   * *Fix:* Replicate identical preprocessing inside Pipelines.

10. **Comparing models on different splits**
    * *Problem:* Changing the `random_state` between model comparisons.
    * *Why:* One model might just get an "easier" split.
    * *Fix:* Lock `random_state`.

11. **Overinterpreting tiny score improvements**
    * *Problem:* Choosing a model that took 10 hours for a 0.001 improve over a model that took 2 seconds.
    * *Why:* Waste of complexity. Simpler is better if scores are identical.
    * *Fix:* Stick to Occam's Razor. 

12. **Assuming the best CV score is always the best real-world model**
    * *Problem:* Being shocked when test performance is slightly lower than CV score.
    * *Why:* CV scores are a *proxy* estimation mechanism.
    * *Fix:* Manage expectations; the unseen test set is the ground truth.


# 25. Interview Questions

Here are 30 crucial interview questions regarding hyperparameter tuning:

1. **What is the difference between a parameter and a hyperparameter?**
   *Parameters are weights learned automatically by the model from the data (e.g., neural network weights). Hyperparameters are configuration values set by the user (e.g., learning rate).*

2. **Why do we need hyperparameter tuning?**
   *To find the optimal model complexity that prevents both underfitting and overfitting on unseen data.*

3. **What is GridSearchCV?**
   *An algorithm that exhaustively tests all possible combinations within a specified parameter grid.*

4. **What is RandomizedSearchCV?**
   *An algorithm that samples a fixed number of parameter combinations randomly from a defined distribution.*

5. **Why must we use Cross-Validation during tuning?**
   *To ensure that hyperparameters perform well across multiple validation sets, avoiding overfitting to a single validation split.*

6. **What does `best_params_` return?**
   *A dictionary containing the parameter setting that yielded the highest mean CV score.*

7. **What does `best_score_` represent?**
   *The highest average cross-validated score achieved by the `best_params_` combination.*

8. **What kind of data does `cv_results_` hold?**
   *Extensive logs of the tuning process, containing the training times, test scores, and statistics for every parameter combination tried.*

9. **Why is the `scoring` parameter vitally important?**
   *Because if you optimize a model for `accuracy` when the dataset is severely imbalanced, the model might learn to just predict the majority class, destroying real-world performance.*

10. **What is a Pipeline in Scikit-Learn?**
    *A tool to chain multiple data transformation steps (like scaling) with a final estimator into a single object.*

11. **How do Pipelines prevent data leakage during Grid Search?**
    *Pipelines ensure transformations (like scaling) are fit *only* on the training fraction of the cross-validation fold, leaving the validation fraction untouched.*

12. **Why shouldn't you tune hyperparameters on the final Test Set?**
    *Because you will overfit to the test set, making it biased and useless for judging actual real-world generalization.*

13. **When is Randomized Search preferred over Grid Search?**
    *When dealing with a vast hyperparameter space (like Random Forest or XGBoost) where exhaustively testing all combinations is computationally impossible.*

14. **What is nested cross-validation?**
    *A technique where an inner CV loop tunes hyperparameters, and an outer CV loop estimates the unbiased error of the tuning procedure itself.*

15. **How can hyperparameter tuning cause overfitting?**
    *If you test too many combinations on small validation sets, by statistical chance, one combination will artificially score very high ("Multiple Comparison Fallacy").*

16. **How should you choose a parameter range for Grid Search?**
    *Use domain knowledge and logarithmically spaced bounds (e.g., `C: [0.1, 1, 10]`) rather than blindly trying linear scales.*

17. **Why use stratified cross-validation for classification?**
    *To ensure each CV fold maintains the original target variable distribution, which is critical for imbalanced classes.*

18. **Why does tuning sometimes result in a model that performs the exact same as baseline?**
    *If the data is easy to separate or Scikit-Learn's mathematically optimal defaults already fit the problem well.*

19. **What hyperparameter mainly controls a Decision Tree's complexity?**
    *`max_depth` (or `min_samples_leaf`).*

20. **Can you parallelize Grid Search?**
    *Yes, by setting `n_jobs=-1`, which tells scikit-learn to utilize all available CPU cores.*

21. **What is data leakage?**
    *When information from outside the training dataset enters the model building process, leading to artificially inflated performance.*

22. **What does the `C` parameter in Logistic Regression control?**
    *Inverse of regularization strength; smaller `C` specifies stronger regularization.*

23. **Why should we scale features for SVM and KNN before tuning?**
    *Because these algorithms rely on distance metrics; unscaled features will improperly dominate the tuning algorithm.*

24. **How does `n_estimators` affect a Random Forest?**
    *Increasing trees generally improves performance but eventually plateaus, just costing more compute without added benefit.*

25. **Is it always necessary to tune hyperparameters?**
    *No. If a baseline model perfectly solves the business problem, tuning is a waste of time.*

26. **What is Bayesian Optimization for tuning?**
    *(Advanced) A strategy that uses probability (like Optuna) to guess the next best parameter combo based on past trials rather than searching randomly.*

27. **What happens if you set `cv=2` in Grid Search?**
    *You perform 2-fold cross-validation. Highly unstable because the model trains on only 50% of the data.*

28. **Does tuning handle missing values?**
    *No. You must impute missing values via a Pipeline step before tuning.*

29. **What happens to the model when you increase `max_features` in a Random Forest?**
    *Trees become more correlated with each other because they are forced to share the same dominant features at splits.*

30. **If you find `max_depth = 10` is best from a grid of `[2, 5, 10]`, what should you do?**
    *You should shift the grid and run again (e.g. `[10, 15, 20]`) because the optimal value might actually lie beyond the border you provided.*


# 26. Quick Revision Cheat Sheet

### Important Concepts

| Concept | Definition |
| :--- | :--- |
| **Parameter** | Core variables the model learns on its own (e.g., regression slope). |
| **Hyperparameter** | Variables you set before training to restrict/guide the model. |
| **Grid Search** | Brute force testing of every specified parameter combination. |
| **Random Search** | Randomly sampling a set number of parameter combinations. |
| **Cross Validation** | Splitting data into rotating train/validate sets to avoid overfitting to one slice. |
| **Scoring** | The specific metric (accuracy, precision, f1) the search algorithm optimizes. |
| **Pipeline** | Chains preprocessing (scaling) and models together to prevent leakage. |
| **Data Leakage** | Occurs if you scale all data before splitting, thus sharing info with validation sets. |
| **Best Parameters** | The optimal hyperparameter combo found during the search. |

### Common Hyperparameters

| Algorithm | Important Hyperparameters |
| :--- | :--- |
| **KNN** | `n_neighbors`, `weights`, `metric` |
| **Logistic Regression** | `C`, `penalty`, `solver` |
| **Decision Tree** | `max_depth`, `min_samples_split`, `min_samples_leaf` |
| **Random Forest** | `n_estimators`, `max_depth`, `max_features` |
| **SVM** | `C`, `gamma`, `kernel` |

### General Tuning Workflow

```text
Baseline Model
   ↓
Choose Right Metric
   ↓
Define Search Space (Grid/Random)
   ↓
Cross Validation Setup (within Pipeline)
   ↓
Execute Search
   ↓
Select Best Parameters
   ↓
Evaluate strictly on untouched Test Set
   ↓
Compare Tuned vs Baseline
```


# 27. Practice Problems

Try these exercises on your own to solidify your knowledge. (No solutions provided). Use `load_wine` or `load_digits` from `sklearn.datasets`.

1. **Manual K:** Manually write a loop to compare different K values (1 to 20) and plot the results.
2. **KNN Grid:** Use `GridSearchCV` on a KNN model, searching `K`, `metric`, and `weights`.
3. **RF Random:** Use `RandomizedSearchCV` for Random Forest (`n_estimators`, `max_depth`).
4. **LogReg Tune:** Tune a Logistic Regression's `C` parameter using `scoring='f1_macro'`.
5. **Tree Depth:** Tune Decision Tree depth using `GridSearchCV`. Plot the relationship between depth and CV score.
6. **Forest Estimators:** Find the specific point where adding more estimators to a Random Forest no longer improves score.
7. **SVM Mastery:** Tune SVM `C` and `gamma` simultaneously.
8. **Pipeline Grid:** Build a `Pipeline` connecting `StandardScaler` and a Decision Tree, then run Grid Search on it.
9. **Impact Analysis:** Compare a baseline SVM to a perfectly tuned SVM. How much did performance improve?
10. **Metric Swapping:** Run Grid Search on a highly imbalanced subset optimized for `accuracy`, then again optimized for `recall`. Compare hyperparameter differences.
11. **CV Results Anatomy:** Convert the `cv_results_` of your Grid Search into a DataFrame and find the *worst* performing parameter combo.
12. **End-to-End Project:** Build an end-to-end hyperparameter tuning workflow for `load_breast_cancer` but use purely `f1` as your deciding metric.


## Next Notebook

`14_imbalance_handling.ipynb`

> Note: The next notebook will focus on dealing with imbalanced datasets. We will cover class imbalance detection, up-sampling/down-sampling (SMOTE), using class weights, and implementing the appropriate evaluation strategies inside cross-validation.
